# Texto


In [1]:
from os import listdir
from lib.texts.getText import get_text
import pymupdf
import json

# Meta

In [36]:
path = "data/pdf/2023/BHA_20230105.pdf"
doc = pymupdf.open(path)
page = doc.load_page(3)
text = page.get_text()

In [110]:
d_mes = {
    "01": "janeiro",
    "02": "fevereiro",
    "03": "março",
    "04": "abril",
    "05": "maio",
    "06": "junho",
    "07": "julho",
    "08": "agosto",
    "09": "setembro",
    "10": "outubro",
    "11": "novembro",
    "12": "dezembro"
}

In [34]:
from lib.helpers.dictClasses import dict_classes
from lib.helpers.slugify import slugify
from lib.helpers.dictTrend import dict_trend

# Current conditions

In [112]:
path = "data/pdf/2023/BHA_20230105.pdf"

def current_conditions(path):
    doc = pymupdf.open(path)
    page = doc.load_page(3)
    text = page.get_text()
    text = text.split("Condições atuais\n")[1]
    text = text.split("\n1\nAbacaxis")[0]
    text = text.replace("\n", "")
    return text

In [122]:
with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)

# analysis

In [1]:
import re
from lib.helpers.slugify import slugify
from lib.helpers.extractFields import extract_fields
from lib.helpers.dictPrognostico import dict_prognostico
from lib.helpers.dictClasses import dict_classes
from lib.helpers.dictTrend import dict_trend
import json
from os import listdir

In [21]:
def clean_text(text: str) -> str:
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
for ed in eds:
    pathPdf = f"data/pdf/2023/BHA_2023{ed}.pdf"
    pathJson = f"data/2023/{ed}.json"
    text = current_conditions(path)
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    pt = {"pt": text}
    boletin['current_conditions'] = pt
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(ed)

In [20]:
def get_bacias(path):
    doc = pymupdf.open(path)
    bacias = {}
    for pg in range(4,15):
        page = doc.load_page(pg)
        blocks = page.get_text_blocks()
        for block in blocks:
            x0, y0, x1, y1, text, block_no, block_type = block
            if x0 < 200 and not "Análise" in text:
                name = clean_text(text)
                slug_name = slugify(name)
                bacias[slug_name] = {}
                bacias[slug_name]['name'] = name
            if x0 > 250 and not "Bacia Amazônica –" in text:
                bacias[slug_name]['text'] = clean_text(text)
    return bacias

In [23]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    pathPdf = f'data/pdf/2023/BHA_2023{ed}.pdf'
    bacias = get_bacias(pathPdf)
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    boletin['bacias'] = bacias
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)

In [25]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    bacias = boletin['bacias']
    analysis = []
    for k, v in bacias.items():
        v['id'] = k
        analysis.append(v)
    boletin['analysis'] = analysis
        
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)

In [15]:
def trans_analysis(analysis):
   for i in analysis:
      text = i['text']
      padrao_mim_max = r"variando entre\s+(\d+(?:[.,]\d+)?)\s+e\s+(\d+(?:[.,]\d+)?)\s+mm"
      match = re.search(padrao_mim_max, text, re.IGNORECASE)
      min = match.group(1)
      max = match.group(2)
      i['min'] = min
      i['max'] = max
      padrao_obs = r"foram observados\s+(\d+(?:[.,]\d+)?\s+mm)"
      match_obs = re.search(padrao_obs, text, re.IGNORECASE)
      observados = match_obs.group(1)
      i['observados'] = observados
      padra_ano = r"o valor de\s+(-?\s?\d+(?:[.,]\d+)?)"
      match_ano = re.search(padra_ano, text, re.IGNORECASE)
      anomalia = match_ano.group(1)
      i['anomalia'] = anomalia
      p_class = r"condição de\s+(.*?)\.\s+Nas próximas semanas"
      match_class = re.search(p_class, text, re.IGNORECASE)
      classification = match_class.group(1)
      # class_slug = slugify(classification)
      # class_es = dict_classes(class_slug, 'es')
      # class_en = dict_classes(class_slug, 'en')
      p_trend = r"comportamento climático indica\s+(.*?)\s+dos volumes de chuva"
      match_trend = re.search(p_trend, text, re.IGNORECASE)
      trend = match_trend.group(1)
      # trend_slug = slugify(trend)
      # trend_es = dict_trend(trend_slug, 'es')
      # trend_en = dict_trend(trend_slug, 'en')
      p_prognostico = r"sugere um comportamento\s+(.*?)(?:\.|$)"
      match_prognostico = re.search(p_prognostico, text, re.IGNORECASE)
      prognostico = match_prognostico.group(1)
      if re.search(r"^de\s", prognostico):
         prognostico = prognostico[3:]    
      # prognostico_slug = slugify(prognostico)
      # prognostico_es = dict_prognostico(prognostico_slug, "es")
      # prognostico_en = dict_prognostico(prognostico_slug, "en")
      i18n = {
               "pt": {
                  "classification": classification,
                  "trend": trend,
                  "prognostico": prognostico
               }
               # ,
               # "en": {
               #    "classification": class_en,
               #    "trend": trend_en,
               #    "prognostico": prognostico_en
               # },
               # "es": {
               #    "classification": class_es,
               #    "trend": trend_es,
               #    "prognostico": prognostico_es
               # }
            }
      i['i18n'] = i18n
   return analysis

In [3]:
path_dir = "data/2023"
eds = listdir(path_dir)
eds = sorted(eds)
eds = [i.split(".")[0] for i in eds]

In [16]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    analysis = boletin['analysis']
    analysis = trans_analysis(analysis)
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    

In [17]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    analysis = boletin['analysis']
    for bacia in analysis:
        del bacia['text']
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    

In [18]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    del boletin['bacias']

    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)

In [ ]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    analysis = boletin['analysis']
    for bacia in analysis:
        del bacia['text']
    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)

# multimodel

In [19]:
def get_seven_days(pathPdf):
    doc = pymupdf.open(pathPdf)
    page = doc.load_page(15)
    blocks = page.get_text_blocks()
    for block in blocks:
        x0, y0, x1, y1, text, block_no, block_type = block
        if y1 > 700:
            text = text.replace("A Figura a esquerda, ", "A Figura ")
            return text
    

In [29]:

pathPdf = "data/pdf/2023/BHA_20230105.pdf"
def get_fourteen_days(pathPdf):
    doc = pymupdf.open(pathPdf)
    page = doc.load_page(16)
    blocks = page.get_text_blocks()
    text = blocks[1][4]
    text = text.replace("A Figura a direita, ", "A Figura ")
    return text  


In [ ]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    pathPdf = f'data/pdf/2023/BHA_2023{ed}.pdf'
    text = get_seven_days(pathPdf)
    multimodel = {
        "pt": {
            "seven_days": text
        }
    }
    
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    
    boletin['multimodel'] = multimodel

    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(ed)

In [ ]:
for ed in eds:
    pathJson = f"data/2023/{ed}.json"
    pathPdf = f'data/pdf/2023/BHA_2023{ed}.pdf'
    text = get_fourteen_days(pathPdf)    
    with open(pathJson, "r") as file:
        boletin = json.load(file)
    
    boletin['multimodel']['pt']['fourteen_days'] = text

    with open(pathJson, "w") as file:
        json.dump(boletin, file, ensure_ascii=False, indent=3)
    print(ed)

# Imagem

In [2]:
import pymupdf 
from lib.images.anomaly import img_anomaly
from lib.images.categorization import img_categorization
from lib.images.reference import img_reference
import os

In [3]:
def img_current_conditions(doc, output_path):

    path = f"{output_path}/current_conditions/"
    page = doc.load_page(3)
    # maps
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['width'] == 800:
            img = i['image'] 
            break
    map = f'{path}map_current_conditions.png'
    with open(map, "wb") as f:
                    f.write(img) # type: ignore
    
    # Tabels
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "1\nAbacaxis" in text:
            break

    top = y0 - 5 # type: ignore
    left = x0 - 15 # type: ignore
    right = x1 + 45 # type: ignore
    bottom = y1 + 5 # type: ignore
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}table_current_conditions.png')

In [ ]:
pathPT = 'data/pdf/2025/BHA_PT_20250101.pdf'

In [ ]:
yyyy = '2023'
for mmdd in eds[1:]:
    output_path = f"data/images/{yyyy}/{mmdd}"
    os.mkdir(output_path)
    os.mkdir(f'{output_path}/analysis')
    os.mkdir(f'{output_path}/anomaly')
    os.mkdir(f'{output_path}/categorization')
    os.mkdir(f'{output_path}/current_conditions')
    os.mkdir(f'{output_path}/multimodel')
    os.mkdir(f'{output_path}/reference')
    print(output_path)

In [13]:
from lib.helpers.slugify import slugify

def img_analysis(doc, output_path, images):
    # bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
    # 'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
    # 'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
    # 'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
    # 'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
    # 'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
    # 'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']
    
    path = f"{output_path}/analysis"
    # images = []
    # for b in bacias:
    #     slug_name = slugify(b)
    #     images.append(f'{slug_name}-acc.png')
    #     images.append(f'{slug_name}-ano.png')
    # for i in range(0,32,3):
    #     bs = bacias[i:i+3]
    #     for j in bs:
    #         slug_name = slugify(j)
    #         images.append(f'{slug_name}-acc.png')
    #     for j in bs:
    #         slug_name = slugify(j)
    #         images.append(f'{slug_name}-ano.png')
    c = 0
    for i in range(4, 15):
        page = doc.load_page(i)
        d = page.get_text("dict")
        blocks = d["blocks"] 
        imgblocks = [b for b in blocks if b["type"] == 1]
        for i in imgblocks:
            # if i['bbox'][0] < 100.0:
            if i['height'] < 3000 and i['height'] > 300:
                img = i['image']
                img_name = images[c]
                with open(f'{path}/{c}-{img_name}', "wb") as f:
                                f.write(img)
                c += 1

In [ ]:
images = [
    'bacia-do-rio-branco-acc.png',
    'bacia-do-rio-negro-acc.png',
    'bacia-do-rio-maranon-acc.png',
    
    'bacia-do-rio-branco-ano.png',
    'bacia-do-rio-negro-ano.png',
    'bacia-do-rio-maranon-ano.png',
    'bacia-do-rio-ucayali-acc.png',
    'bacia-do-rio-napo-acc.png',
    'curso-principal-do-rio-amazonas-peru-acc.png',
    'bacia-do-rio-ucayali-ano.png',
    'bacia-do-rio-napo-ano.png',
    'curso-principal-do-rio-amazonas-peru-ano.png',
    'bacia-do-rio-javari-acc.png',
    'bacia-do-rio-ica-putumayo-acc.png',
    'bacia-do-rio-jutai-acc.png',
    'bacia-do-rio-javari-ano.png',
    'bacia-do-rio-ica-putumayo-ano.png',
    'bacia-do-rio-jutai-ano.png',
    'bacia-do-rio-jurua-acc.png',
    'bacia-do-rio-japura-caqueta-acc.png',
    'bacia-do-rio-tefe-acc.png',
    'bacia-do-rio-jurua-ano.png',
    'bacia-do-rio-japura-caqueta-ano.png',
    'bacia-do-rio-tefe-ano.png',
    'bacia-do-rio-coari-acc.png',
'bacia-do-rio-purus-acc.png',
'curso-principal-do-rio-solimoes-acc.png',
 'bacia-do-rio-coari-ano.png',
 'bacia-do-rio-purus-ano.png',
 'curso-principal-do-rio-solimoes-ano.png',
 'bacia-dos-rios-beni-e-madre-de-dios-acc.png',
 'bacia-do-rio-mamore-acc.png',
 'bacia-dos-rios-beni-e-madre-de-dios-ano.png',
 'bacia-do-rio-guapore-itenez-acc.png',
 
 'bacia-do-rio-mamore-ano.png',
 'bacia-do-rio-guapore-itenez-ano.png',
 'bacia-do-rio-ji-parana-acc.png',
 'bacia-do-rio-aripuana-acc.png',
 'bacia-do-rio-madeira-acc.png',
 'bacia-do-rio-ji-parana-ano.png',
 'bacia-do-rio-aripuana-ano.png',
 'bacia-do-rio-madeira-ano.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-amazonas-ano.png',
 'bacia-do-rio-abacaxis-acc.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-amazonas-acc.png',
 'bacia-do-rio-abacaxis-ano.png',
 'bacia-do-rio-juruena-acc.png',
 
 
 'bacia-do-rio-juruena-ano.png',
 'bacia-do-rio-teles-pires-acc.png',
 'bacia-do-rio-teles-pires-ano.png',
 'bacia-do-rio-tapajos-acc.png',
 
 'bacias-da-margem-esquerda-do-rio-amazonas-noroeste-do-para-acc.png',
 
 'bacia-do-rio-tapajos-ano.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-noroeste-do-para-ano.png',
 'bacia-do-rio-curua-una-acc.png',
 'bacia-do-rio-curua-una-ano.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-nordeste-do-pa-acc.png',
 'bacias-da-margem-esquerda-do-rio-amazonas-nordeste-do-pa-ano.png',
 'bacia-do-rio-iriri-ano.png',
 'bacia-do-rio-iriri-acc.png',
 
 
 
 'bacia-do-rio-xingu-acc.png',
 'curso-principal-do-rio-amazonas-brasil-acc.png',
 
 'curso-principal-do-rio-amazonas-brasil-ano.png',
 'bacia-do-rio-xingu-ano.png']

In [16]:
doc = pymupdf.open('data/pdf/2025/BHA_PT_20250108.pdf')
img_analysis(doc, 'test', images)

In [4]:
def img_multimodel(doc, output_path):
    path = f"{output_path}/multimodel"
    page = doc.load_page(15)
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['height'] == 800:
            img = i
            break
    left, top, right, bottom = img['bbox'] # type: ignore
    right += 100
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/seven_days.png')
    
    # fourteen_days
    page = doc.load_page(16)
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['height'] == 800:
            img = i
            break
    left, top, right, bottom = img['bbox'] # type: ignore
    right += 100
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/fourteen_days.png')

In [5]:
def get_images(pathPT, output_path):
    doc = pymupdf.open(pathPT)
    img_current_conditions(doc, output_path)
    img_analysis(doc, output_path, images)
    img_multimodel(doc, output_path)
    img_reference(doc, output_path)
    img_categorization(doc, output_path)
    img_anomaly(doc, output_path)

In [10]:
pathPT = 'data/pdf/2025/BHA_PT_20250108.pdf'
output_path = 'data/images/2025/0108'
get_images(pathPT, output_path)

In [ ]:
pathPT = f'data/pdf/2023/BHA_20230108.pdf'
output_path = f"data/images/{yyyy}/0105"
doc = pymupdf.open(pathPT)
def img_multimodel_2023(doc, output_path):
    page = doc.load_page(15)
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    c = 0
    img_name = ["seven_days", "fourteen_days"]
    for i in imgblocks:
        if i['height'] == 800:
            img = i['image']
            name = img_name[c]
            with open(f'{output_path}/multimodel/{name}.png', "wb") as f:
                f.write(img)
    
            c += 1



In [ ]:
for mmdd in eds[2:]:
    pathPT = f'data/pdf/2023/BHA_2023{mmdd}.pdf'
    output_path = f"data/images/{yyyy}/{mmdd}"
    doc = pymupdf.open(pathPT)
    # img_multimodel_2023(doc, output_path)
    img_multimodel(doc, output_path)
    print(mmdd)

In [ ]:
for mmdd in eds[1:]:
    pathPT = f'data/pdf/2023/BHA_2023{mmdd}.pdf'
    output_path = f"data/images/{yyyy}/{mmdd}"
    doc = pymupdf.open(pathPT)
    img_reference(doc, output_path)
    print(mmdd)

In [ ]:
for mmdd in eds:
    pathPT = f'data/pdf/2023/BHA_2023{mmdd}.pdf'
    output_path = f"data/images/{yyyy}/{mmdd}"
    doc = pymupdf.open(pathPT)
    img_categorization(doc, output_path)
    print(mmdd)
    

In [48]:
yyyy = '2024'
mmdd = '1106'
pathPT = f"data/pdf/2024/BHA_PT_{yyyy}{mmdd}.pdf"
pathEN = f"data/pdf/2024/BHA_EN_{yyyy}{mmdd}.pdf"
pathES = f"data/pdf/2024/BHA_ES_{yyyy}{mmdd}.pdf"
output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mmdd}"

get_images(pathPT, output_path)

In [50]:
doc = pymupdf.open(pathPT)
img_analysis(doc, output_path)

In [51]:
import os

In [66]:
eds = os.listdir("data/pdf/2024")
pt = []
for i in eds:
    if "PT" in i:
        pt.append(i)
eds = sorted(pt)


In [ ]:
yyyy = '2024'
for ed in eds:
    mmdd = ed[11:15]
    pathPT = f"data/pdf/2024/BHA_PT_{yyyy}{mmdd}.pdf"
    # pathEN = f"data/pdf/2024/BHA_EN_{yyyy}{mmdd}.pdf"
    # pathES = f"data/pdf/2024/BHA_ES_{yyyy}{mmdd}.pdf"
    output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mmdd}"
    get_images(pathPT, output_path)
    print(mmdd)

In [59]:
path = 'data/images/2024'
eds = os.listdir(path)
for ed in eds:
    fullPath = f'{path}/{ed}'
    multimodel_path = f'{fullPath}/multimodel'
    analysis = os.listdir(multimodel_path)
    for i in analysis:
        file = f'{multimodel_path}/{i}'
        os.remove(file)
        # print(file)

In [143]:
from cloudinary import uploader, config, api
import json

In [144]:
config(
  cloud_name="dveg6vhbi",
  api_key="954532419258163",
  api_secret="MiQDehQG2afKCxVi-QC8qMFE0Ag"
)

In [145]:
yyyy = 2023 
for mmdd in eds:
    pathPT = f'data/pdf/2023/BHA_2023{mmdd}.pdf'
    d_imgs = {}
    output_path = f'data/images/2023/{mmdd}'
    for section in os.listdir(output_path):
        full_path = f'{output_path}/{section}'
        d_imgs[section] = {}
        imgs = os.listdir(full_path)
        for img in imgs:
            full_img_path = f'{full_path}/{img}'
            img_name = img.split(".")[0]
            result = uploader.upload(
                full_img_path,
                public_id=img_name,
                folder=f"clima-amazonia/{yyyy}/{mmdd}/{section}",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
            url = result['secure_url']
            d_imgs[section][img_name] = url
            print(result)
    print(output_path)
    
    pdfs = {}
    result = uploader.upload(
                pathPT,
                public_id=pathPT.split("/")[-1].replace(".pdf",""),
                folder=f"clima-amazonia/{yyyy}/{mmdd}/",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
    pdfs['pt'] = result['secure_url']
    
    jsonPath = f'./data/{yyyy}/{mmdd}.json'
    with open(jsonPath, 'r') as f:
        bulletin_dict = json.load(f)
    bulletin_dict['images'] = d_imgs
    bulletin_dict['pdf'] = pdfs
    with open(jsonPath, 'w') as f:
            json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)
    


{'asset_id': 'f6d91bd3cb15331dcd12f03731c55ed3', 'public_id': 'clima-amazonia/2023/0105/current_conditions/table_current_conditions', 'version': 1774467555, 'version_id': '9ff06295e356bc031a058e9fe0236763', 'signature': '466f5e260fa7a2e1457fd9781ff0322f670251e3', 'width': 1415, 'height': 260, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-25T19:39:15Z', 'tags': [], 'bytes': 69218, 'type': 'upload', 'etag': '78c1285b45419a3b8435c917735baac8', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1774467555/clima-amazonia/2023/0105/current_conditions/table_current_conditions.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774467555/clima-amazonia/2023/0105/current_conditions/table_current_conditions.png', 'asset_folder': 'clima-amazonia/2023/0105/current_conditions', 'display_name': 'table_current_conditions', 'original_filename': 'table_current_conditions', 'api_key': '954532419258163'}
{'asset_id': 'bb720155cfa45226a

In [ ]:
yyyy = "2024"
for ed in eds[1:]:
    mmdd = ed[11:15]
    number = f"{yyyy}{mmdd}"
    pathPT = f"/home/inacio/script-clima-amazonia/data/pdf/{yyyy}/BHA_PT_{number}.pdf"
    pathEN = f"/home/inacio/script-clima-amazonia/data/pdf/{yyyy}/BHA_EN_{number}.pdf"
    pathES = f"/home/inacio/script-clima-amazonia/data/pdf/{yyyy}/BHA_ES_{number}.pdf"
    d_imgs = {}
    output_path = f"/home/inacio/script-clima-amazonia/data/images/{yyyy}/{mmdd}"
    for section in os.listdir(output_path):
        full_path = f'{output_path}/{section}'
        d_imgs[section] = {}
        imgs = os.listdir(full_path)
        for img in imgs:
            full_img_path = f'{full_path}/{img}'
            img_name = img.split(".")[0]
            result = uploader.upload(
                full_img_path,
                public_id=img_name,
                folder=f"clima-amazonia/{yyyy}/{mmdd}/{section}",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
            url = result['secure_url']
            d_imgs[section][img_name] = url
            print(result)
    print(output_path)
    pdfs = {}
    result = uploader.upload(
                pathPT,
                public_id=pathPT.split("/")[-1].replace(".pdf",""),
                folder=f"clima-amazonia/{yyyy}/{mmdd}/",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
    pdfs['pt'] = result['secure_url']

    result = uploader.upload(
                pathEN,
                public_id=pathEN.split("/")[-1].replace(".pdf",""),
                folder=f"clima-amazonia/{yyyy}/{mmdd}/",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
    pdfs['en'] = result['secure_url']

    result = uploader.upload(
                pathES,
                public_id=pathES.split("/")[-1].replace(".pdf",""),
                folder=f"clima-amazonia/{yyyy}/{mmdd}/",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
    pdfs['es'] = result['secure_url']
    jsonPath = f'./data/{yyyy}/{mmdd}.json'
    with open(jsonPath, 'r') as f:
        bulletin_dict = json.load(f)
    bulletin_dict['images'] = d_imgs
    bulletin_dict['pdf'] = pdfs
    with open(jsonPath, 'w') as f:
            json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)

In [81]:
pathPT = f"/home/inacio/script-clima-amazonia/data/pdf/{yyyy}/BHA_PT_{number}.pdf"
pathEN = f"/home/inacio/script-clima-amazonia/data/pdf/{yyyy}/BHA_EN_{number}.pdf"
pathES = f"/home/inacio/script-clima-amazonia/data/pdf/{yyyy}/BHA_ES_{number}.pdf"

In [88]:
result = uploader.upload(
                pathES,
                public_id=pathPT.split("/")[-1].replace(".pdf",""),
                folder=f"clima-amazonia/{yyyy}/{mmdd}/",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )

In [89]:
pdfs['es'] = result['secure_url']

In [91]:
jsonPath = f'./data/{yyyy}/{mmdd}.json'
with open(jsonPath, 'r') as f:
    bulletin_dict = json.load(f)
bulletin_dict['images'] = d_imgs
bulletin_dict['pdf'] = pdfs
with open(jsonPath, 'w') as f:
        json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)

# Issues

In [3]:
from os import listdir

In [ ]:
path = "data/pdf/2023"
for pdf in sorted(listdir(path)):
    mmdd = pdf[8:12]
    path_pdf = f"{path}/{pdf}"
    doc = pymupdf.open(path_pdf)
    page = doc.load_page(0)
    pix = page.get_pixmap()
    pix.save(f"test/{mmdd}.png")
    
    print(mmdd)

In [ ]:
path = "data/2023"
eds = sorted(listdir(path))
d_n = {}
c = 1
for i in eds:
    mmdd = i.split(".")[0]
    mm = mmdd[:2]
    if mm in d_n:
        d_i = {
            "number": c,
            "url": f"2023/{mmdd}",
            "img": ""
        }
        d_n[mm]["issues"].append(d_i)
    else:
        d_n[mm] = {
             "month": mm,
             "issues": []
        }
        d_i = {
            "number": c,
            "url": f"2024/{mmdd}",
            "img": ""
        }
        d_n[mm]["issues"].append(d_i)
    c += 1
    print(mmdd)

In [19]:
ns = [v for v in d_n.values()]

In [21]:
from cloudinary import uploader, config, api

In [22]:
config(
  cloud_name="dveg6vhbi",
  api_key="954532419258163",
  api_secret="MiQDehQG2afKCxVi-QC8qMFE0Ag"
)

In [24]:
for n in ns:
    issues = n['issues']
    for i in issues:
        url = i['url']
        mmdd = url.split("/")[1]
        path_img = f"test/{mmdd}.png"
        result = uploader.upload(
                path_img,
                public_id=mmdd,
                folder=f"clima-amazonia/2023/{mmdd}",
                use_filename=True,
                overwrite=True,
                unique_filename=True
            )
        secure_url = result['secure_url']
        i['img'] = secure_url
        print(result)

{'asset_id': '46bec83a5fb67b50edaade4c188eb8d8', 'public_id': 'clima-amazonia/2023/0105/0105', 'version': 1774530260, 'version_id': '76600b79cb9c932ca98ec809cad7186d', 'signature': 'a2996a35f6a29b599699b3c78f830cc4a87a5941', 'width': 612, 'height': 792, 'format': 'png', 'resource_type': 'image', 'created_at': '2026-03-26T13:04:20Z', 'tags': [], 'bytes': 62787, 'type': 'upload', 'etag': '6ae6586051ddbbb10eb116469f0402bb', 'placeholder': False, 'url': 'http://res.cloudinary.com/dveg6vhbi/image/upload/v1774530260/clima-amazonia/2023/0105/0105.png', 'secure_url': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774530260/clima-amazonia/2023/0105/0105.png', 'asset_folder': 'clima-amazonia/2023/0105', 'display_name': '0105', 'original_filename': '0105', 'api_key': '954532419258163'}
{'asset_id': 'dacd4299f1dc8842075b04e66922a625', 'public_id': 'clima-amazonia/2023/0112/0112', 'version': 1774530261, 'version_id': '5dc99076027d8787f0757868df2c0c51', 'signature': '927be34029b0f97ed5dc1a9e81

In [25]:
with open("test/numbers.json", "w") as file:
    json.dump(ns, file, indent=3)